<div style="background: linear-gradient(135deg, #0a1628 0%, #1a365d 50%, #2d4a7a 100%); padding: 40px 30px; border-radius: 16px; margin-bottom: 24px; border-left: 6px solid #e6a817;">
  <h1 style="color: #e6a817; font-size: 2em; margin: 0;">🔍 Lab 04: Trace Your Advisory Agent</h1>
  <p style="color: #c5d5ea; font-size: 1.15em; margin-top: 10px;">Zava Bank Workshop — Microsoft Foundry Agent Observability</p>
</div>

<div style="background-color: #0d1f3c; border-left: 5px solid #e6a817; padding: 20px 24px; border-radius: 10px; margin-bottom: 20px;">

## 📘 What You'll Learn

Instrument the Zava Bank Client Advisor agent with **OpenTelemetry tracing** — console output for local debugging, Azure Monitor for production observability.

You'll see exactly what happens inside the agent on every request:
- Client queries entering the system
- Tool invocations and function calls
- Model calls and token usage
- Response generation and latency

**Part A** — Console tracing for rapid local iteration  
**Part B** — Azure Monitor tracing for production audit trails

</div>

<div style="background-color: #1a1a2e; border-left: 5px solid #c9a227; padding: 20px 24px; border-radius: 10px; margin-bottom: 20px;">

## 🏦 The Zava Bank Story

The advisory agent is working well in dev. Advisors like the portfolio analysis, risk reviews come back clean, the team is optimistic.

Then a compliance officer files an incident report:

> *"An advisor asked about a conservative client's risk profile and the agent recommended aggressive growth positions. We need to understand what happened inside."*

Without tracing, this is a **black box**. You can see the input and the output, but nothing in between — no record of which tools fired, what the model reasoned, or how the final answer was assembled.

In regulated financial services, that's not acceptable. You need an **audit trail** that captures the full execution path of every advisory interaction.

That's what this lab builds.

</div>

---

## Part A: Console Tracing (Local Debugging)

### Step 1 — Configure Environment

Load credentials and enable GenAI tracing flags. These environment variables tell the Azure SDK to emit detailed spans for every model call and capture full message content.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

# Enable GenAI tracing in the Azure SDK
os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"
os.environ["OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT"] = "true"

print("✅ Environment configured — GenAI tracing enabled")

### Step 2 — Configure OpenTelemetry with ConsoleSpanExporter

This wires up OpenTelemetry to dump every span to stdout. Useful during development — you can see the full trace tree right in your notebook output.

In [ ]:
from azure.core.settings import settings

settings.tracing_implementation = "opentelemetry"

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter
from azure.ai.projects.telemetry import AIProjectInstrumentor

span_exporter = ConsoleSpanExporter()
tracer_provider = TracerProvider()
tracer_provider.add_span_processor(SimpleSpanProcessor(span_exporter))
trace.set_tracer_provider(tracer_provider)

tracer = trace.get_tracer("zava-bank-tracing")

AIProjectInstrumentor().instrument()

print("✅ Console tracing active — spans will appear in cell output")

### Step 3 — Create the AIProjectClient and Agent

Connect to your Foundry project and create a traced version of the Zava Bank Advisor agent.

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.agents.models import PromptAgentDefinition
from openai import AzureOpenAI

project_endpoint = os.environ["PROJECT_ENDPOINT"]
model_name = os.environ.get("MODEL_DEPLOYMENT_NAME", "gpt-4o")

project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=DefaultAzureCredential(),
)

# Create a traced version of the advisory agent
agent = project_client.agents.create_version(
    agent_name="zava-bank-advisor-traced",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions=(
            "You are the Zava Bank Client Advisor. Help financial advisors with "
            "portfolio analysis, risk assessment, and market research. "
            "Always include compliance disclaimers."
        ),
    ),
)

print(f"🏦 Agent created: {agent.name} (v{agent.version})")
print(f"   Model: {model_name}")

In [ ]:
# Set up the OpenAI client for conversation calls
openai_client = project_client.get_openai_client()

print("✅ OpenAI client ready")

### Step 4 — Run a Traced Advisory Query

Watch the console output carefully — you'll see spans for the HTTP call, model inference, and response assembly.

In [ ]:
with tracer.start_as_current_span("zava-advisory-query"):
    conversation = openai_client.conversations.create()

    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={
            "agent_reference": {
                "name": agent.name,
                "type": "agent_reference",
            }
        },
        input="What should I know about portfolio concentration risk for our conservative clients?",
    )

    print(f"\n🏦 Advisor Response:\n{response.output_text}")

<div style="background-color: #1a2332; border-left: 4px solid #4a9eff; padding: 16px 20px; border-radius: 8px; margin: 16px 0;">

### 🔎 Reading the Console Trace

In the output above, you should see structured span data:

| Field | What It Tells You |
|-------|-------------------|
| `name` | The operation — e.g., `zava-advisory-query`, HTTP calls, model inference |
| `trace_id` | Groups all spans from a single request into one trace |
| `parent_id` | Shows the parent-child relationship between spans |
| `attributes` | Operation-specific metadata — model name, token counts, HTTP status |
| `start_time` / `end_time` | Latency for each operation |

This is the audit trail. If the compliance officer asks "what happened inside?", you can reconstruct the full execution path from these spans.

</div>

---

## Part B: Azure Monitor Tracing (Production Observability)

Console tracing is good for debugging. But in production you need traces persisted, searchable, and correlated across services. Azure Monitor gives you that.

### Step 5 — Shutdown Console Tracer, Configure Azure Monitor

In [ ]:
# Flush and shut down the console exporter
tracer_provider.shutdown()

from azure.monitor.opentelemetry import configure_azure_monitor

app_insights_connection_string = (
    project_client.telemetry.get_application_insights_connection_string()
)

configure_azure_monitor(connection_string=app_insights_connection_string)

tracer = trace.get_tracer("zava-bank-tracing")

print("✅ Azure Monitor tracing active")
print(f"   Traces will appear in Application Insights")

### Step 6 — Run a Traced Query with Custom Financial Attributes

Custom span attributes let you filter traces by business dimensions — client segment, risk tolerance, query type. This is what makes tracing useful for compliance, not just debugging.

In [ ]:
with tracer.start_as_current_span("zava-advisory-risk-query") as span:
    # Custom attributes for financial context
    span.set_attribute("finance.client_segment", "high_net_worth")
    span.set_attribute("finance.risk_tolerance", "conservative")
    span.set_attribute("finance.query_type", "risk_review")
    span.set_attribute("zava.agent_name", agent.name)

    conversation = openai_client.conversations.create()

    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={
            "agent_reference": {
                "name": agent.name,
                "type": "agent_reference",
            }
        },
        input=(
            "A high-net-worth client with conservative risk tolerance holds 60% of their "
            "portfolio in a single tech stock. What are the risks and what rebalancing "
            "approach would you recommend?"
        ),
    )

    # Capture the trace ID for portal lookup
    trace_id = span.get_span_context().trace_id

    print(f"\n🏦 Advisor Response:\n{response.output_text}")
    print(f"\n📊 Trace ID: {format(trace_id, '032x')}")
    print("   → Use this ID to find the full trace in Azure Monitor")

### Step 7 — More Custom Attributes: Market Outlook Query

Different query type, different client segment. The custom attributes let you slice traces by any dimension that matters to your compliance and ops teams.

In [ ]:
with tracer.start_as_current_span("zava-advisory-market-query") as span:
    span.set_attribute("finance.client_segment", "institutional")
    span.set_attribute("finance.query_type", "market_outlook")
    span.set_attribute("finance.asset_class", "equities")
    span.set_attribute("zava.agent_name", agent.name)
    span.set_attribute("zava.agent_version", str(agent.version))

    conversation = openai_client.conversations.create()

    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={
            "agent_reference": {
                "name": agent.name,
                "type": "agent_reference",
            }
        },
        input=(
            "What are the current equity market conditions that institutional clients "
            "should be aware of? Focus on sector rotation signals and volatility trends."
        ),
    )

    trace_id = span.get_span_context().trace_id

    print(f"\n🏦 Advisor Response:\n{response.output_text}")
    print(f"\n📊 Trace ID: {format(trace_id, '032x')}")

<div style="background-color: #1a2332; border-left: 4px solid #4a9eff; padding: 16px 20px; border-radius: 8px; margin: 16px 0;">

### 🖥️ Viewing Traces in Azure Portal

1. Open the [Azure Portal](https://portal.azure.com)
2. Navigate to your **Application Insights** resource (linked to your Foundry project)
3. Go to **Transaction search** and paste a Trace ID from above
4. Or go to **Logs** and run a KQL query:

```kql
dependencies
| where name contains "zava-advisory"
| where customDimensions["finance.client_segment"] == "high_net_worth"
| project timestamp, name, duration, 
         customDimensions["finance.risk_tolerance"],
         customDimensions["finance.query_type"]
| order by timestamp desc
```

The custom attributes (`finance.client_segment`, `finance.risk_tolerance`, `finance.query_type`) are now queryable dimensions. A compliance team can filter to: *"Show me all traces where a conservative client got an aggressive recommendation."*

</div>

---

## Cleanup

In [ ]:
project_client.agents.delete_agent(agent_name=agent.name)
print(f"🧹 Deleted agent: {agent.name}")

<div style="background: linear-gradient(135deg, #0a1628 0%, #1a365d 50%, #2d4a7a 100%); padding: 30px 24px; border-radius: 14px; margin-top: 24px; border-left: 6px solid #e6a817;">

## 📋 Summary

| Approach | When to Use | What You Get |
|----------|------------|-------------|
| **Console tracing** | Local dev / debugging | Immediate span output in notebook cells |
| **Azure Monitor** | Staging / production | Persistent, searchable traces in Application Insights |
| **Custom attributes** | Always | Business-specific filtering — client segment, risk tolerance, query type |

### Custom Attributes Used in This Lab

| Attribute | Purpose |
|-----------|---------|
| `finance.client_segment` | Filter by client type (HNW, institutional, retail) |
| `finance.risk_tolerance` | Correlate recommendations with client risk profile |
| `finance.query_type` | Categorize interactions (risk review, market outlook, portfolio rebalance) |
| `finance.asset_class` | Track which asset classes are being discussed |
| `zava.agent_name` | Identify which agent version handled the request |

### Key Takeaway

In regulated industries, tracing is **compliance infrastructure**, not optional observability. When a compliance officer asks *"what happened inside?"*, you need an answer — not a shrug.

</div>